# Recipe Classification and Embedding Visualization

In this part of the notebook, we demonstrate how to classify recipes into pre-fixed categories using zero-shot classification, followed by embedding each recipe into a semantic vector space. We then visualize these embeddings in 3D to explore the natural groupings of recipes based on their predicted labels.

The goal is to validate the hypothesis that **zero-shot classification models** can organize data points coherently within the chosen categories, allowing us to observe well-defined clusters based solely on the classification predictions.

## Setup
Import general libraries.

In [ ]:
import pandas as pd
import numpy as np
import time
import ast
import random
import os

## Load data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/rcp-nlp/Mauro

/content/drive/.shortcut-targets-by-id/1ii6eI76RNWTvQduHWw9n3IAzswUbvxRG/rcp-nlp/Mauro


In [ ]:
def load_data(filepath, sample_size=None):
    """
    Load the RecipeNLG dataset from CSV

    Args:
        filepath: Path to the full_dataset.csv
        sample_size: Number of recipes to sample (None to load all)

    Returns:
        pandas DataFrame with the recipe data
    """
    start_time = time.time()

    # Define the converters to parse stringified lists
    converters = {
        'ingredients': ast.literal_eval,
        'directions': ast.literal_eval,
        'ner': ast.literal_eval
    }

    # For a quick analysis, we can load a sample of the data
    if sample_size is not None:
        # Count the total number of lines in the file
        with open(filepath, 'r', encoding='utf-8') as f:
            line_count = sum(1 for _ in f)

        # Generate random line indices to sample
        skip_indices = sorted(random.sample(range(1, line_count), line_count - sample_size - 1))
        df = pd.read_csv(filepath, skiprows=skip_indices, converters=converters)
    else:
        # Load the full dataset
        df = pd.read_csv(filepath, converters=converters)

    # Ensure proper types
    # df['id'] = df['id'].astype(int)

    print(f"Data loaded in {time.time() - start_time:.2f} seconds")
    print(f"Dataset shape: {df.shape}")

    return df

We use 10,000 samples for classification to balance between computational cost and obtaining representative results.

In [ ]:
df_labels_path = '/content/drive/MyDrive/rcp-nlp/Mauro/df_labels.csv'

if not os.path.exists(df_labels_path):
  DATASET_PATH = '/content/drive/MyDrive/rcp-nlp/full_dataset.csv'
  df = load_data(DATASET_PATH, sample_size=10000)

For this task, we load a pre-trained model (`facebook/bart-large-mnli`) with the zero-shot classification pipeline, which allows us to classify data without any task-specific training.

In [ ]:
from transformers import pipeline


classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=0)

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Device set to use cuda:0


## Zero-Shot Classification

We construct a unified text representation for each recipe by combining its title, ingredients, and directions, and then define a set of candidate labels such as appetizer, main dish, and dessert.

Then, the zero-shot model is applied to this dataset to predict the most likely label for each recipe.

In [ ]:
from datasets import Dataset


# Define candidate labels
candidate_labels = ["appetizer", "main dish", "side dish","dessert", "drink", "snack"]

# Combine title, ingredients, directions
def join_text(row):
    return f"Title: {row['title']}\nIngredients: {', '.join(row['ingredients'])}\nDirections: {', '.join(row['directions'])}"


if os.path.exists(df_labels_path):
  df_labels = pd.read_csv(df_labels_path)
else:
  # Prepare Dataset
  df['text'] = df.apply(join_text, axis=1)
  dataset = Dataset.from_pandas(df)

  # Batch prediction
  results = classifier(dataset["text"], candidate_labels)

  # Extract top label
  semantic_labels = [r["labels"][0] for r in results]

  # Build result DataFrame
  df_labels = pd.DataFrame({
      "title": df["title"],
      "semantic_label": semantic_labels,
      "recipe": df['text']
  })

df_labels

,title,semantic_label,recipe,x,y,z
0,Cabbage Rolls,side dish,Title: Cabbage Rolls\nIngredients: 1 head cabb...,8.303648,0.370874,8.667727
1,Baked Cheese Sandwiches,snack,Title: Baked Cheese Sandwiches\nIngredients: 5...,7.518969,2.218069,7.326159
2,Easy Chicken Cacciatore,main dish,Title: Easy Chicken Cacciatore\nIngredients: 1...,11.227042,2.108896,8.157785
3,Pumpkin Bread,snack,Title: Pumpkin Bread\nIngredients: 3 1/2 c. fl...,1.599275,0.347783,5.267959
4,Mixed Vegetable Bake,main dish,Title: Mixed Vegetable Bake\nIngredients: 1 (8...,9.113039,0.488083,8.329683
...,...,...,...,...,...,...
9995,Curried Mango,snack,"Title: Curried Mango\nIngredients: 15 ml oil, ...",10.377009,1.180990,9.803973
9996,Hamburger Casserole,main dish,Title: Hamburger Casserole\nIngredients: 1 lb ...,9.654181,2.034361,7.729259
9997,Buffalo Chicken Lasagna,main dish,Title: Buffalo Chicken Lasagna\nIngredients: 1...,9.936378,2.537367,5.585028
9998,Pecan Coated Cheeseball,snack,Title: Pecan Coated Cheeseball\nIngredients: 1...,5.928112,2.407698,8.915957


Now, we use the `all-mpnet-base-v2` model from Sentence Transformers to convert each recipe into a numerical embedding. Each embedding is a 768-dimensional vector that captures the semantic meaning of the recipe. These embeddings will be used for visualization and clustering.

In [ ]:
from sentence_transformers import SentenceTransformer

print("Generating embeddings...")

embedder = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

# Assuming your combined recipe text column is named "recipe"
embeddings = embedder.encode(df_labels["recipe"].tolist(), show_progress_bar=True)

Generating embeddings...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.4k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/313 [00:00<?, ?it/s]

In [ ]:
embeddings.shape

(10000, 768)

UMAP reduces the 768-dimensional embeddings to 3D while preserving semantic structure. We then plot the results, coloring each point by its predicted label to reveal recipe groupings.

In [ ]:
import plotly.express as px
import umap


print("Reducing to 3D...")
reducer = umap.UMAP(n_components=3, metric="cosine")
embeddings_3d = reducer.fit_transform(embeddings)

# Add 3D coordinates to df_labels (not subset_df)
df_labels["x"] = embeddings_3d[:, 0]
df_labels["y"] = embeddings_3d[:, 1]
df_labels["z"] = embeddings_3d[:, 2]

# 3D visualization on the labeled subset
fig = px.scatter_3d(
    df_labels,
    x="x", y="y", z="z",
    color="semantic_label",
    hover_data=["title"]
)
fig.update_layout(title="3D Recipe Clusters by Semantic Label")
fig.show()

Reducing to 3D...


/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


## Results evaluation
The visualization demonstrates that zero-shot classification combined with semantic embeddings can effectively organize recipes by category.

While natural overlaps appear (especially between categories that commonly coexist, such as main and side dishes) other categories like desserts form distinctly separated clusters. Categories with broader or less-defined roles, such as snacks and drinks, are more dispersed throughout the embedding space.

Overall, this approach provides meaningful insights into the semantic structure of recipe data, capturing both clear groupings and the nuanced relationships between different types of recipes.